[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/03_data_analysis_and_eda/90_exercises.ipynb)

# Bài tập — Chương 03: Data Analysis & EDA (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/03_data_analysis_and_eda

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from ml_course.checks import check_close, check_equal, check_true
from ml_course.utils import repo_path

churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
fraud = pd.read_csv(repo_path("data", "sample", "fraud_transactions.csv"))

## L0 — Nhớ khái niệm

**Câu 1.** Phân loại các dữ liệu sau là *có cấu trúc* hay *không có cấu trúc*: (a) bảng giao dịch ngân hàng; (b) email khách hàng; (c) ảnh X-quang; (d) file log máy chủ dạng text; (e) bảng điểm học sinh.

✍️ Trả lời:

**Đáp án:** có cấu trúc: (a), (e). Không có cấu trúc: (b), (c). (d) là **bán cấu trúc** (semi-structured) — có định dạng một phần, cần parse để thành bảng.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích MCAR, MAR, MNAR, mỗi loại một ví dụ trong bài toán dự đoán giá nhà.

✍️ Trả lời:

**Đáp án:** MCAR — thiếu ngẫu nhiên hoàn toàn (lỗi hệ thống mất ngẫu nhiên 3% dòng). MAR — thiếu phụ thuộc biến quan sát được (tin đăng ở một website cụ thể không có trường "tuổi nhà"). MNAR — thiếu phụ thuộc chính giá trị bị thiếu (người bán nhà rất cũ cố tình không ghi tuổi nhà).

## L2 — Tính tay

**Câu 3.** Dữ liệu: `[2, 4, 4, 5, 6, 7, 8, 30]`. Với $Q_1 = 4$, $Q_3 = 7.25$ (theo `np.percentile`), tính IQR, cận dưới và cận trên của quy tắc 1.5×IQR, và số outlier.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
iqr = 3.25
lower = 4 - 1.5 * 3.25
upper = 7.25 + 1.5 * 3.25
n_outliers = 1
### END CODE HERE ###

In [ ]:
data = np.array([2, 4, 4, 5, 6, 7, 8, 30])
q1, q3 = np.percentile(data, [25, 75])
check_close([iqr, lower, upper], [q3 - q1, q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)], message="IQR và hai cận")
check_equal(n_outliers, int(((data < lower) | (data > upper)).sum()), "số outlier")

## L3 — Code

**Bài 4.** Viết `profile(df)` trả về DataFrame (index = tên cột) với các cột: `dtype` (chuỗi), `pct_missing` (%), `n_unique`, `is_constant` (True nếu chỉ có ≤ 1 giá trị khác nhau, không tính NaN).

In [ ]:
def profile(df):
    ### START CODE HERE ### (≈ 6 dòng)
    result = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_missing": df.isna().mean() * 100,
        "n_unique": df.nunique(),
        "is_constant": df.nunique() <= 1,
    })
    ### END CODE HERE ###
    return result


toy = pd.DataFrame({"a": [1, 2, None, 4], "b": ["x", "x", "x", "x"], "c": [1.0, 2.0, 3.0, 4.0]})
profile(toy)

In [ ]:
p = profile(toy)
check_close(p.loc["a", "pct_missing"], 25.0, message="pct_missing cột a")
check_equal(p["is_constant"].tolist(), [False, True, False], "phát hiện cột hằng")
check_equal(p.loc["c", "n_unique"], 4, "n_unique cột c")

**Bài 5.** Viết `build_vocab(texts, min_count)` trả về dict `word → index` với `"<UNK>"` có index 0, các từ (chữ thường, tách bằng regex `\w+`) xuất hiện **≥ min_count** lần được đánh số từ 1 theo thứ tự **tần suất giảm dần** (hoà thì theo thứ tự chữ cái). Và `encode(text, vocab)` trả về list index (từ lạ → 0).

In [ ]:
from collections import Counter


def build_vocab(texts, min_count=1):
    ### START CODE HERE ### (≈ 4 dòng)
    counts = Counter(w for t in texts for w in re.findall(r"\w+", t.lower()))
    words = sorted((w for w, c in counts.items() if c >= min_count), key=lambda w: (-counts[w], w))
    vocab = {"<UNK>": 0}
    vocab.update({w: i + 1 for i, w in enumerate(words)})
    ### END CODE HERE ###
    return vocab


def encode(text, vocab):
    ### START CODE HERE ### (≈ 1 dòng)
    ids = [vocab.get(w, 0) for w in re.findall(r"\w+", text.lower())]
    ### END CODE HERE ###
    return ids


texts = ["Hàng tốt, giao nhanh", "hàng tốt", "giao chậm quá", "Hàng lỗi"]
vocab = build_vocab(texts, min_count=2)
print(vocab, encode("hàng rất tốt", vocab))

In [ ]:
check_equal(vocab, {"<UNK>": 0, "hàng": 1, "giao": 2, "tốt": 3}, "vocab với min_count=2")
check_equal(encode("Hàng rất tốt", vocab), [1, 0, 3], "encode với từ lạ")

## L4 — Debug

**Bài 6.** Một đồng nghiệp xây model phát hiện gian lận, AUC trên tập test = 0.999 với feature `chargeback_amount` (số tiền khách hàng yêu cầu hoàn lại **sau khi** phát hiện giao dịch gian lận). Mô phỏng tình huống và đo AUC của **riêng** feature đó. Giải thích vấn đề.

In [ ]:
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
fraud_sim = fraud.copy()
fraud_sim["chargeback_amount"] = np.where(fraud_sim["is_fraud"] == 1, fraud_sim["amount_vnd"], 0) + rng.normal(0, 1000, len(fraud_sim))
### START CODE HERE ### (≈ 2 dòng)
auc_leaky = roc_auc_score(fraud_sim["is_fraud"], fraud_sim["chargeback_amount"])
auc_amount = roc_auc_score(fraud_sim["is_fraud"], fraud_sim["amount_vnd"])
### END CODE HERE ###
print(f"AUC chargeback_amount = {auc_leaky:.3f} | AUC amount_vnd = {auc_amount:.3f}")

In [ ]:
check_true(auc_leaky > 0.95 and auc_amount < auc_leaky, "feature rò rỉ có AUC 'quá tốt'")

✍️ Giải thích:

**Đáp án:** `chargeback_amount` chỉ được biết **sau** khi giao dịch đã bị xác định gian lận → tại thời điểm cần dự đoán (lúc giao dịch xảy ra) giá trị này **chưa tồn tại**. Đây là **target leakage**: điểm offline gần hoàn hảo nhưng model vô dụng khi triển khai. Phải loại feature này; nguyên tắc: chỉ dùng thông tin có sẵn tại thời điểm dự đoán.

## L5 — Phân tích

**Bài 7.** Dữ liệu `admissions` (mô phỏng tuyển sinh) cho thấy tỉ lệ đỗ của nữ thấp hơn nam khi gộp lại. Tính tỉ lệ đỗ **theo từng khoa** và **tổng hợp**, rồi giải thích.

In [ ]:
admissions = pd.DataFrame({
    "khoa": ["Kỹ thuật"] * 2 + ["Y"] * 2,
    "giới": ["nam", "nữ"] * 2,
    "nộp": [800, 100, 200, 700],
    "đỗ": [480, 70, 40, 175],
})
### START CODE HERE ### (≈ 3 dòng)
admissions["tỉ_lệ"] = admissions["đỗ"] / admissions["nộp"]
total = admissions.groupby("giới")[["đỗ", "nộp"]].sum()
overall_rate = total["đỗ"] / total["nộp"]
### END CODE HERE ###
print(admissions)
print(overall_rate)

In [ ]:
check_true(overall_rate["nữ"] < overall_rate["nam"], "tổng hợp: nữ thấp hơn")
by_dept = admissions.pivot(index="khoa", columns="giới", values="tỉ_lệ")
check_true((by_dept["nữ"] > by_dept["nam"]).all(), "theo từng khoa: nữ CAO hơn ở mọi khoa → nghịch lý Simpson")

✍️ Giải thích:

**Đáp án:** Nữ chủ yếu nộp vào khoa Y — khoa có tỉ lệ đỗ thấp (cạnh tranh hơn); nam nộp chủ yếu vào Kỹ thuật — tỉ lệ đỗ cao. "Khoa" là biến gây nhiễu: ảnh hưởng tới cả giới tính người nộp lẫn khả năng đỗ. Trong từng khoa, nữ có tỉ lệ đỗ cao hơn. (Đây là cấu trúc của vụ UC Berkeley 1973 nổi tiếng.)

## L6 — Thực tế

**Bài 8.** Thực hiện EDA ngắn trên `fraud_transactions.csv`. Viết hàm `fraud_rate_by(df, col)` trả về DataFrame (index = giá trị của `col`) gồm `n` và `fraud_rate`, sắp xếp `fraud_rate` giảm dần. Áp dụng cho `merchant_category` và nhóm giờ `night = (hour <= 4) | (hour >= 23)`. Ghi 3 insight.

In [ ]:
def fraud_rate_by(df, col):
    ### START CODE HERE ### (≈ 3 dòng)
    out = df.groupby(col)["is_fraud"].agg(n="size", fraud_rate="mean")
    out = out.sort_values("fraud_rate", ascending=False)
    ### END CODE HERE ###
    return out


print("Tỉ lệ gian lận chung:", round(fraud["is_fraud"].mean(), 4))
by_cat = fraud_rate_by(fraud, "merchant_category")
by_night = fraud_rate_by(fraud.assign(night=(fraud["hour"] <= 4) | (fraud["hour"] >= 23)), "night")
print(by_cat, "\n", by_night)

In [ ]:
check_equal(list(by_cat.columns), ["n", "fraud_rate"], "đúng cột")
check_true(by_cat["fraud_rate"].is_monotonic_decreasing, "sắp xếp giảm dần")
check_true(by_cat.index[0] in {"electronics", "online_games"}, "nhóm rủi ro nhất là electronics/online_games")
check_true(by_night.loc[True, "fraud_rate"] > by_night.loc[False, "fraud_rate"], "ban đêm rủi ro hơn")

✍️ 3 insight:

**Gợi ý:** (1) Tỉ lệ gian lận chỉ ~1% → dữ liệu rất mất cân bằng, accuracy vô nghĩa (chương 19). (2) `electronics` và `online_games` có tỉ lệ gian lận cao nhất — hàng dễ bán lại / tài sản số. (3) Giao dịch ban đêm, ở nước ngoài, số tiền lớn, nhiều giao dịch trong 24h đều liên quan tới gian lận cao hơn → là các feature tốt; cân nhắc tạo tương tác (vd: đêm × nước ngoài).

## L7 — Nâng cao

**Bài 9.** Kiểm tra cơ chế missing **MCAR vs MAR** một cách định lượng: tạo dữ liệu trong đó `age_years` bị thiếu **nhiều hơn ở nhà xa trung tâm** (MAR). Viết `missingness_test(df, col, other)` dùng **Kolmogorov–Smirnov test** (`scipy.stats.ks_2samp`) so sánh phân phối của `other` giữa nhóm *thiếu* và *không thiếu* `col`; trả về p-value. p-value nhỏ ⇒ việc thiếu phụ thuộc vào `other` (không phải MCAR).

In [ ]:
from ml_course.datasets import make_house_prices

h = make_house_prices(n=3000, seed=1)
rng = np.random.default_rng(1)
p_missing = np.clip(h["distance_to_center_km"] / h["distance_to_center_km"].max(), 0, 1) * 0.4   # xa → dễ thiếu
h_mar = h.copy()
h_mar.loc[rng.random(len(h)) < p_missing, "age_years"] = np.nan
h_mcar = h.copy()
h_mcar.loc[rng.random(len(h)) < 0.15, "age_years"] = np.nan


def missingness_test(df, col, other):
    ### START CODE HERE ### (≈ 2 dòng)
    miss = df[col].isna()
    p_value = stats.ks_2samp(df.loc[miss, other], df.loc[~miss, other]).pvalue
    ### END CODE HERE ###
    return p_value


p_mar = missingness_test(h_mar, "age_years", "distance_to_center_km")
p_mcar = missingness_test(h_mcar, "age_years", "distance_to_center_km")
print(f"p-value MAR = {p_mar:.2e} | p-value MCAR = {p_mcar:.3f}")

In [ ]:
check_true(p_mar < 1e-6, "phát hiện được MAR (p rất nhỏ)")
check_true(p_mcar > 0.01, "không bác bỏ MCAR khi dữ liệu thật sự MCAR")